# Ollama Full Sweep

This notebook runs the full generation x embedding sweep against the Ollama-backed clinical RAG pipeline.


In [1]:
from __future__ import annotations

import os
import sys
from pathlib import Path

import pandas as pd

repo_root = Path.cwd()
if not (repo_root / "main.py").exists():
    for parent in Path.cwd().resolve().parents:
        if (parent / "main.py").exists():
            repo_root = parent
            break

sys.path.insert(0, str(repo_root))

from eval.embedding_benchmark import model_slug
from eval.model_benchmark import ModelBenchmarkConfig, run_model_benchmark
from helpers.experiment_models import EMBEDDING_MODEL_SWEEP, GENERATION_MODEL_SWEEP

os.environ.setdefault("GENERATION_MODEL_PROVIDER", "ollama")
os.environ.setdefault("VERIFIER_MODEL_PROVIDER", "ollama")
os.environ.setdefault("REASONER_MODEL_PROVIDER", "ollama")
os.environ.setdefault("RAG_EMBEDDING_MODEL_PROVIDER", "ollama")
os.environ.setdefault("OLLAMA_ENDPOINT", "http://10.0.0.201:8000")
os.environ.setdefault("INPUT_BASE_DIR", str(repo_root / "data" / "evidence" / "mimic_discharge_subset"))
os.environ.setdefault("OUTPUT_BASE_DIR", str(repo_root / "output"))

generation_model_sweep = list(GENERATION_MODEL_SWEEP)
embedding_model_sweep = list(EMBEDDING_MODEL_SWEEP)
sample_size = 25
shared_index_root = repo_root / "output" / "benchmark_indexes"
use_umls = os.environ.get("UMLS_ENABLED", "true").strip().lower() == "true"
schema_guided = os.environ.get("INDEX_SCHEMA_GUIDED", "false").strip().lower() == "true"
mimic_csv = repo_root / "data" / "mimic_iv_note" / "discharge.csv"

print("OLLAMA_ENDPOINT:", os.environ["OLLAMA_ENDPOINT"])
print("generation_model_sweep:", generation_model_sweep)
print("embedding_model_sweep:", embedding_model_sweep)
print("sample_size:", sample_size)
print("use_umls:", use_umls)
print("schema_guided:", schema_guided)


OLLAMA_ENDPOINT: http://10.0.0.201:8000
generation_model_sweep: ['gemma4', 'qwen3.5:9b', 'medgemma1.5']
embedding_model_sweep: ['qwen3-embedding:0.6b', 'embeddinggemma:latest', 'all-minilm:latest']
sample_size: 5
use_umls: True
schema_guided: False


In [ ]:
rows = []
for embedding_model in embedding_model_sweep:
    output_root = repo_root / "output" / "ollama_full_sweep" / model_slug(embedding_model)
    results = await run_model_benchmark(
        ModelBenchmarkConfig(
            input_dir=Path(os.environ["INPUT_BASE_DIR"]),
            output_root=output_root,
            index_root=shared_index_root,
            generation_models=tuple(generation_model_sweep),
            embedding_model=embedding_model,
            use_umls=use_umls,
            schema_guided=schema_guided,
            mimic_csv=(mimic_csv if mimic_csv.exists() else None),
            sample_size=sample_size,
        )
    )
    rows.extend(result.__dict__ for result in results)

results_df = pd.DataFrame(rows).sort_values(["embedding_model", "mean_top_cosine_similarity", "mean_cosine_similarity", "exact_match", "mean_query_seconds"], ascending=[True, False, False, False, True])
results_df


LlamaIndex index is missing or incompatible:
  - missing /Users/oluwatosinoso/Library/CloudStorage/OneDrive-hull.ac.uk/argumentation_schemes/output/ollama_full_sweep/qwen3-embedding_0_6b/gemma4/index_manifest.json
